In [4]:
INDIR = '/home/lcastri/git/causal-sim2real/HRISim_docker/src/HRISim/postprocessing/hrisim_postprocess/csv'
BAGNAME = 'EXP_BATTERY_03'
TOD = 'H1'

In [ ]:
import json
import math
import xml.etree.ElementTree as ET
import os
from matplotlib import pyplot as plt
import pandas as pd
import numpy as np

def readScenario():
    # Load and parse the XML file
    tree = ET.parse('/home/lcastri/git/causal-sim2real/HRISim_docker/src/pedsim_ros/pedsim_simulator/scenarios/warehouse.xml')
    root = tree.getroot()
    
    tmp = {}
    for waypoint in root.findall('waypoint'):
        waypoint_id = waypoint.get('id')
        x = float(waypoint.get('x'))
        y = float(waypoint.get('y'))
        r = float(waypoint.get('r'))
        tmp[waypoint_id] = {'x': x, 'y': y, 'r': r}
    return tmp

def pathLength(task):
    return np.sum([
        math.sqrt(
            (WPS_COORD[task['path'][wp_idx]]['x'] - WPS_COORD[task['path'][wp_idx+1]]['x'])**2 + 
            (WPS_COORD[task['path'][wp_idx]]['y'] - WPS_COORD[task['path'][wp_idx+1]]['y'])**2
        )
        for wp_idx in range(len(task['path'])-1)])


WPS_COORD = readScenario()

with open(os.path.join(INDIR, BAGNAME, f'tasks-{TOD}.json')) as pkl_file:
    TASKS = json.load(pkl_file)
TASK_IDs = list(range(TASKS['n_tasks']))
        
df = pd.read_csv(os.path.join(INDIR, BAGNAME, f"{BAGNAME}.csv"))


DTASKS = {task_id: {'path_length': 0,
                   'duration': 0,
                   'battery_cost_actual': 0,
                   'battery_cost_bn': 0,
                   'battery_cost_cm': 0} for task_id in TASK_IDs}
for task_id in TASK_IDs:
    
    start_time = TASKS[str(task_id)]['start']
    end_time = TASKS[str(task_id)]['end']
                
    # Filter the data for the task based on the start and end time
    task_df = df[(df['ros_time'] >= start_time) & (df['ros_time'] <= end_time)]
    
    ## path length
    DTASKS[task_id]['path_length'] = pathLength(TASKS[str(task_id)])
    
    ## actual battery consumption
    battery = task_df['R_B'].to_numpy()
    DTASKS[task_id]['battery_cost_actual'] = battery[0] - battery[-1]
    
    ## estimated battery consumption BN
    DTASKS[task_id]['battery_cost_bn'] = TASKS[str(task_id)]['battery_cost_bn']
    
    # estimated battery consumption CM
    DTASKS[task_id]['battery_cost_cm'] = TASKS[str(task_id)]['battery_cost_cm']
    
    ## duration
    DTASKS[task_id]['duration'] = end_time - start_time
    
    if battery[0] - battery[-1] < -5 or battery[0] - battery[-1] == 0: 
        print(f"Removing task {task_id} with battery consumption {battery[0] - battery[-1]}")
        del DTASKS[task_id]
    
print(DTASKS)

gt = []
bn = []
cm = []
for task_id, task_info in DTASKS.items():
    gt.append(task_info['battery_cost_actual'])
    bn.append(task_info['battery_cost_bn'])
    cm.append(task_info['battery_cost_cm'])
    mae_bn = np.mean(np.abs(np.array(gt) - np.array(bn)))
    mae_cm = np.mean(np.abs(np.array(gt) - np.array(cm)))
    std_bn = np.std(np.abs(np.array(gt) - np.array(bn)))
    std_cm = np.std(np.abs(np.array(gt) - np.array(cm)))

# plt.figure(figsize=(12, 6))
# plt.bar(['BN', 'CM'], [mae_bn, mae_cm], yerr=[std_bn, std_cm], capsize=10)


ae_38_bn = np.abs(DTASKS[38]['battery_cost_actual'] - DTASKS[38]['battery_cost_bn'])
ae_38_cm = np.abs(DTASKS[38]['battery_cost_actual'] - DTASKS[38]['battery_cost_cm'])
ae_48_bn = np.abs(DTASKS[48]['battery_cost_actual'] - DTASKS[48]['battery_cost_bn'])
ae_48_cm = np.abs(DTASKS[48]['battery_cost_actual'] - DTASKS[48]['battery_cost_cm'])

print(f"Task 38 - BN AE: {ae_38_bn:.4f}, CM AE: {ae_38_cm:.4f}")
print(f"Task 48 - BN AE: {ae_48_bn:.4f}, CM AE: {ae_48_cm:.4f}")
print(f"Combined Task Path Length: {(DTASKS[38]['path_length'] + DTASKS[48]['path_length']):.4f}")
print(f"Combined Task Duration: {(DTASKS[38]['duration'] + DTASKS[48]['duration']):.4f}")
print(f"Combined Task GT: {(DTASKS[38]['battery_cost_actual'] + DTASKS[48]['battery_cost_actual']):.4f}")
print(f"Combined Task AE -- BN AE: {(ae_38_bn + ae_48_bn):.4f}, CM AE: {(ae_38_cm + ae_48_cm):.4f}")


{0: {'path_length': 21.375305786795508, 'duration': 87.39600000000019, 'battery_cost_actual': 0.6674346923828267, 'battery_cost_bn': 0.29862165451049805, 'battery_cost_cm': 0.325745053589344}, 1: {'path_length': 21.56288109237248, 'duration': 74.69699999999966, 'battery_cost_actual': 0.47797393798828125, 'battery_cost_bn': 0.12798070907592773, 'battery_cost_cm': 0.13960502296686172}, 2: {'path_length': 17.85, 'duration': 51.36299999999983, 'battery_cost_actual': 0.21269989013671875, 'battery_cost_bn': 0.3839421272277832, 'battery_cost_cm': 0.4188150689005852}, 3: {'path_length': 21.774167887782653, 'duration': 91.04700000000003, 'battery_cost_actual': 0.8180084228515625, 'battery_cost_bn': 0.08532047271728516, 'battery_cost_cm': 0.09307001531124115}, 4: {'path_length': 20.809820624195552, 'duration': 82.45200000000023, 'battery_cost_actual': 1.115821838378892, 'battery_cost_bn': 0.3839421272277832, 'battery_cost_cm': 0.4188150689005852}, 5: {'path_length': 18.823491154701024, 'duration